# Solución · Gestor de pedidos y stock

Este notebook resuelve las cuatro consignas de "Para expandir" del programa base:

1. Reponer stock con `reponer_stock()` y reprocesar los pedidos rechazados por falta de existencias.
2. Descuentos por cantidad mediante un método de `Pedido`.
3. Informe del producto más vendido.
4. Reemplazo de algunos pedidos de prueba por objetos creados con datos ingresados por `input()`.

Las funciones `buscar_producto()`, `procesar_pedido()` y `calcular_facturacion()` del programa base no cambian: se reutilizan tal cual, tanto en el procesamiento inicial como en el reproceso posterior a reponer stock.

## 1. Clases `Producto` y `Pedido` extendidas

Cambios respecto del programa base:

- `Producto` suma `unidades_vendidas` y el método `registrar_venta()`, que `Pedido.confirmar()` llama automáticamente. Así el conteo de ventas vive en el objeto que corresponde, sin variables sueltas por fuera.
- `resumen_stock()` en `Producto` arma la línea de reporte (stock + ventas) como texto reutilizable.
- `Pedido` incorpora `TRAMOS_DESCUENTO`, un atributo de clase con pares `(cantidad_mínima, porcentaje)`, y `calcular_porcentaje_descuento()`, que devuelve el porcentaje del primer tramo alcanzado. `confirmar()` lo aplica al calcular `self.total` y guarda `self.porcentaje_descuento` y `self.descuento` para poder mostrarlos después.
- `resumen()` en `Pedido` arma la línea de reporte de un pedido (incluyendo el descuento, si tuvo) como texto reutilizable, igual que `resumen_stock()` en `Producto`.
- `reponer_stock()` no cambia: ya estaba en el programa base, solo faltaba usarlo (se usa en la sección 4).

In [ ]:
class Producto:
    def __init__(
        self,
        codigo: str,
        nombre: str,
        precio: float,
        stock: int,
    ) -> None:
        self.codigo: str = codigo
        self.nombre: str = nombre
        self.precio: float = precio
        self.stock: int = stock
        self.unidades_vendidas: int = 0

    def tiene_stock(self, cantidad: int) -> bool:
        return cantidad > 0 and cantidad <= self.stock

    def descontar_stock(self, cantidad: int) -> bool:
        if not self.tiene_stock(cantidad):
            return False

        self.stock -= cantidad
        return True

    def reponer_stock(self, cantidad: int) -> None:
        if cantidad > 0:
            self.stock += cantidad

    def registrar_venta(self, cantidad: int) -> None:
        self.unidades_vendidas += cantidad

    def resumen_stock(self) -> str:
        return (
            f"{self.codigo} · {self.nombre:<22} "
            f"{self.stock:>3} unidades   "
            f"({self.unidades_vendidas} vendidas)"
        )


class Pedido:
    TRAMOS_DESCUENTO: list[tuple[int, float]] = [
        (10, 0.15),
        (5, 0.10),
        (3, 0.05),
    ]

    def __init__(
        self,
        numero: int,
        codigo_producto: str,
        cantidad: int,
    ) -> None:
        self.numero: int = numero
        self.codigo_producto: str = codigo_producto
        self.cantidad: int = cantidad
        self.estado: str = "Pendiente"
        self.total: float = 0.0
        self.porcentaje_descuento: float = 0.0
        self.descuento: float = 0.0

    def calcular_porcentaje_descuento(self) -> float:
        # Devuelve el porcentaje del primer tramo que la cantidad alcanza.
        for cantidad_minima, porcentaje in self.TRAMOS_DESCUENTO:
            if self.cantidad >= cantidad_minima:
                return porcentaje
        return 0.0

    def confirmar(self, producto: Producto) -> None:
        subtotal: float = producto.precio * self.cantidad
        self.porcentaje_descuento = self.calcular_porcentaje_descuento()
        self.descuento = subtotal * self.porcentaje_descuento
        self.total = subtotal - self.descuento
        self.estado = "Confirmado"
        producto.registrar_venta(self.cantidad)

    def rechazar(self, motivo: str) -> None:
        self.estado = f"Rechazado: {motivo}"
        self.total = 0.0
        self.porcentaje_descuento = 0.0
        self.descuento = 0.0

    def resumen(self) -> str:
        detalle_descuento: str = (
            f" (desc. {self.porcentaje_descuento:.0%})"
            if self.porcentaje_descuento > 0
            else ""
        )
        return (
            f"#{self.numero:<3} {self.codigo_producto:<4} "
            f"x {self.cantidad:<3} {self.estado:<34} "
            f"${self.total:>10,.2f}{detalle_descuento}"
        )

## 2. Funciones de búsqueda, procesamiento y reposición

`buscar_producto()`, `procesar_pedido()` y `calcular_facturacion()` son las mismas del programa base. Se agregan tres funciones:

- `reponer_y_reprocesar()`: repone stock según un diccionario `{código: cantidad}` y vuelve a pasar por `procesar_pedido()` (sin modificarla) solo a los pedidos rechazados por `"stock insuficiente"`. Los rechazados por otro motivo, como `"producto inexistente"`, quedan igual.
- `contar_confirmados()`: cuenta pedidos confirmados, para no repetir ese bucle en el informe.
- `producto_mas_vendido()`: recorre la lista de productos con `max()` sobre `unidades_vendidas` y devuelve `None` si todavía no se vendió nada.

In [ ]:
def buscar_producto(
    productos: list[Producto],
    codigo: str,
) -> Producto | None:
    for producto in productos:
        if producto.codigo == codigo:
            return producto

    return None


def procesar_pedido(
    pedido: Pedido,
    productos: list[Producto],
) -> None:
    producto: Producto | None = buscar_producto(
        productos,
        pedido.codigo_producto,
    )

    if producto is None:
        pedido.rechazar("producto inexistente")
        return

    if producto.descontar_stock(pedido.cantidad):
        pedido.confirmar(producto)
    else:
        pedido.rechazar("stock insuficiente")


def reponer_y_reprocesar(
    procesados: list[Pedido],
    productos: list[Producto],
    reposiciones: dict[str, int],
) -> list[Pedido]:
    for codigo, cantidad in reposiciones.items():
        producto: Producto | None = buscar_producto(productos, codigo)
        if producto is not None:
            producto.reponer_stock(cantidad)

    reprocesados: list[Pedido] = []

    for pedido in procesados:
        if pedido.estado == "Rechazado: stock insuficiente":
            procesar_pedido(pedido, productos)
            reprocesados.append(pedido)

    return reprocesados


def calcular_facturacion(pedidos: list[Pedido]) -> float:
    total: float = 0.0

    for pedido in pedidos:
        if pedido.estado == "Confirmado":
            total += pedido.total

    return total


def contar_confirmados(pedidos: list[Pedido]) -> int:
    total: int = 0

    for pedido in pedidos:
        if pedido.estado == "Confirmado":
            total += 1

    return total


def producto_mas_vendido(productos: list[Producto]) -> Producto | None:
    if not productos:
        return None

    top: Producto = max(productos, key=lambda producto: producto.unidades_vendidas)

    if top.unidades_vendidas <= 0:
        return None

    return top

## 3. Creación de los objetos

Los pedidos 1, 2 y 5 quedan fijos, para que el escenario de stock insuficiente sea siempre el mismo. Los pedidos 3 y 4 usan el mismo patrón `MODO_DEMO` de la solución del programa 1: con `MODO_DEMO = True` se arman con datos fijos, y con `MODO_DEMO = False` se piden por `input()`.

In [ ]:
MODO_DEMO: bool = True

productos: list[Producto] = [
    Producto("L01", "Lámpara modular", 48_000.0, 8),
    Producto("M02", "Mesa auxiliar", 92_000.0, 3),
    Producto("P03", "Panel acústico", 36_500.0, 10),
]

if MODO_DEMO:
    datos_pedido_3: tuple[int, str, int] = (3, "P03", 6)
    datos_pedido_4: tuple[int, str, int] = (4, "X99", 1)
else:
    datos_pedido_3 = (
        int(input("Número del pedido 3: ")),
        input("Código de producto del pedido 3: "),
        int(input("Cantidad del pedido 3: ")),
    )
    datos_pedido_4 = (
        int(input("Número del pedido 4: ")),
        input("Código de producto del pedido 4: "),
        int(input("Cantidad del pedido 4: ")),
    )

pendientes: list[Pedido] = [
    Pedido(1, "L01", 2),
    Pedido(2, "M02", 4),
    Pedido(*datos_pedido_3),
    Pedido(*datos_pedido_4),
    Pedido(5, "L01", 10),
]

procesados: list[Pedido] = []

## 4. Procesamiento inicial

Mismo bucle `while` del programa base: mientras haya pedidos pendientes, se procesan y se guardan en `procesados`. Con el stock inicial, el pedido 2 (Mesa auxiliar x4, stock 3) y el pedido 5 (Lámpara modular x10, stock insuficiente tras el pedido 1) quedan rechazados por falta de existencias.

In [ ]:
while pendientes:
    pedido_actual: Pedido = pendientes.pop(0)
    procesar_pedido(pedido_actual, productos)
    procesados.append(pedido_actual)

print("PEDIDOS PROCESADOS (antes de reponer stock)")
print("-" * 68)

for pedido in procesados:
    print(pedido.resumen())

## 5. Reposición de stock y reproceso

Acá se usa `reponer_stock()` (a través de `reponer_y_reprocesar()`) para reponer stock de los productos que lo necesitan, y se reintentan únicamente los pedidos que se habían rechazado por falta de existencias. El pedido rechazado por "producto inexistente" no se toca.

In [ ]:
reposiciones: dict[str, int] = {"M02": 5, "L01": 10}
reprocesados: list[Pedido] = reponer_y_reprocesar(procesados, productos, reposiciones)

print(f"Stock repuesto en: {list(reposiciones.keys())}")
print(f"Pedidos reprocesados: {[pedido.numero for pedido in reprocesados]}")
print("-" * 68)

for pedido in reprocesados:
    print(pedido.resumen())

## 6. Informe final

In [ ]:
print("PEDIDOS - ESTADO FINAL")
print("-" * 68)

for pedido in procesados:
    print(pedido.resumen())

print("\nSTOCK Y VENTAS POR PRODUCTO")
print("-" * 68)

for producto in productos:
    print(producto.resumen_stock())

facturacion: float = calcular_facturacion(procesados)
confirmados: int = contar_confirmados(procesados)
top: Producto | None = producto_mas_vendido(productos)

print("\nRESUMEN")
print(f"Pedidos confirmados: {confirmados}")
print(f"Facturación total: ${facturacion:,.2f}")

if top is not None:
    print(f"Producto más vendido: {top.nombre} ({top.unidades_vendidas} unidades)")
else:
    print("Todavía no se vendió ningún producto.")

## Conclusiones

- **Reponer stock y reprocesar**: `reponer_y_reprocesar()` llama a `reponer_stock()` sobre los productos del diccionario y vuelve a pasar por `procesar_pedido()` (sin modificarla) solo a los pedidos rechazados por `"stock insuficiente"`. En esta corrida, los pedidos #2 (Mesa auxiliar) y #5 (Lámpara modular) pasan de rechazados a confirmados; el pedido #4, rechazado por `"producto inexistente"`, queda igual porque reponer stock no resuelve ese motivo.
- **Descuento por cantidad**: `calcular_porcentaje_descuento()` recorre `TRAMOS_DESCUENTO` de mayor a menor umbral. Con las cantidades usadas se ven los tres tramos: el pedido de 4 unidades queda con 5%, el de 6 unidades con 10%, y el de 10 unidades con 15%; el de 2 unidades no llega al primer tramo y no tiene descuento.
- **Producto más vendido**: `producto_mas_vendido()` compara `unidades_vendidas` entre productos con `max()`, un contador que cada `Producto` actualiza solo mediante `registrar_venta()` al confirmarse un pedido. En esta corrida es la Lámpara modular, con 12 unidades vendidas entre el pedido inicial y el reprocesado.
- **Datos por `input()`**: los pedidos 3 y 4 se arman según `MODO_DEMO`, replicando el patrón demo/interactivo de la solución del programa 1, para poder alternar entre datos fijos y datos cargados por consola sin duplicar código.